# Day 10: End-to-End Orchestration & Verification

This notebook simulates a complete E2E run of a synthetic patient from raw JSON through the various stages to Gold metrics.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, concat, lit, expr, monotonically_increasing_id
import duckdb

spark = SparkSession.builder \
    .appName("Day10_E2E") \
    .getOrCreate()

# Step 1: Raw Patient Ingestion (Bronze)
raw_data = [
    ("pat_999", "999-99-9999", "1950-01-01", "INDEX_CONDITION", "2023-05-01"),
    ("pat_999", "999-99-9999", "1950-01-01", "FOLLOWUP", "2023-05-15")
]
raw_df = spark.createDataFrame(raw_data, ["patient_id", "ssn", "dob", "condition", "date"])
print("1. Bronze Raw:")
raw_df.show()


In [ ]:
# Step 2: De-Identification
salt = "e2e_salt"
deid_df = raw_df.withColumn("ssn_fpe", sha2(concat(col("ssn"), lit(salt)), 256)) \
                .withColumn("dob_shifted", expr("date_add(cast(dob as date), cast(abs(hash(patient_id)) % 61 as int) - 30)")) \
                .select("patient_id", "ssn_fpe", "dob_shifted", "condition", "date")
print("2. De-Identified:")
deid_df.show(truncate=False)


In [ ]:
# Step 3: OMOP Silver
omop_df = deid_df.withColumn("condition_occurrence_id", monotonically_increasing_id()) \
                 .withColumn("person_id", col("patient_id")) \
                 .withColumn("condition_start_date", col("date")) \
                 .withColumn("condition_source_value", col("condition")) \
                 .select("condition_occurrence_id", "person_id", "condition_start_date", "condition_source_value")
print("3. OMOP Silver:")
omop_df.show()


In [ ]:
# Step 4: Data Quality (Mock pass)
# Assuming all pass here
print("4. DQ Passed. Ready for Gold.")


In [ ]:
# Step 5: Gold Metrics (via DuckDB simulating Snowflake dbt)
con = duckdb.connect(database=':memory:')
omop_pd = omop_df.toPandas()
con.execute("CREATE TABLE silver_conditions AS SELECT * FROM omop_pd")

gold_query = """
WITH index_admissions AS (
    SELECT person_id, CAST(condition_start_date AS DATE) AS index_date
    FROM silver_conditions
    WHERE condition_source_value = 'INDEX_CONDITION'
),
subsequent_admissions AS (
    SELECT person_id, CAST(condition_start_date AS DATE) AS subsequent_date
    FROM silver_conditions
)
SELECT 
    i.person_id,
    i.index_date,
    COUNT(s.subsequent_date) AS readmission_count_30d
FROM index_admissions i
LEFT JOIN subsequent_admissions s 
    ON i.person_id = s.person_id 
    AND s.subsequent_date > i.index_date 
    AND (s.subsequent_date - i.index_date) <= 30
GROUP BY i.person_id, i.index_date
"""
print("5. Gold Mart - Readmissions (Expect 1 readmission for pat_999):")
print(con.execute(gold_query).df())
